# Two carts, three springs: which motions are solutions?

The demo: two carts of mass $m$ on a track, joined to the walls and to each other by three springs of
spring constant $k$ (wall - spring - cart 1 - spring - cart 2 - spring - wall). $x_1$ and $x_2$ are how
far each cart is to the right of where it sits at rest.

Newton's second law for each cart (the middle spring is stretched by $x_2 - x_1$):

$$m\,x_1'' = -k\,x_1 + k\,(x_2 - x_1) \qquad\qquad m\,x_2'' = -k\,x_2 - k\,(x_2 - x_1)$$

We let SymPy do the plugging-in: is a pair of functions $x_1(t)$, $x_2(t)$ a solution, and is the sum of
two solutions a solution?

## 1. Setup

In [ ]:
%matplotlib inline
from sympy import *
init_printing()

t = symbols('t', real=True)
k, m = symbols('k m', positive=True)
w0 = sqrt(k/m)

## 2. The test: plug in and see what is left over

`leftover(x1, x2)` moves everything in each equation to one side and simplifies. A pair of functions is a
solution exactly when **both** leftovers are 0 for every $t$.

In [ ]:
def leftover(x1, x2):
    eq1 = m*diff(x1, t, 2) - (-k*x1 + k*(x2 - x1))
    eq2 = m*diff(x2, t, 2) - (-k*x2 - k*(x2 - x1))
    return simplify(eq1), simplify(eq2)

## 3. Some candidates

A guess that is **not** a solution: cart 1 oscillates at $\omega_0 = \sqrt{k/m}$ and cart 2 stays put.

In [ ]:
leftover(cos(w0*t), 0)

A janky one that **is** a solution: two different frequencies mixed together.

In [ ]:
jank1_x1 = cos(w0*t) + cos(sqrt(3)*w0*t)
jank1_x2 = cos(w0*t) - cos(sqrt(3)*w0*t)
leftover(jank1_x1, jank1_x2)

Another janky one, with different amounts of each piece.

In [ ]:
jank2_x1 = 3*cos(w0*t) - 2*cos(sqrt(3)*w0*t)
jank2_x2 = 3*cos(w0*t) + 2*cos(sqrt(3)*w0*t)
leftover(jank2_x1, jank2_x2)

## 4. Add two solutions: still a solution?

The equations are linear and homogeneous (every term has exactly one $x_1$ or $x_2$ in it, to the first
power), so the sum of two solutions should be one too. Check the sum, and then any amounts $a$ and $b$
of each.

In [ ]:
leftover(jank1_x1 + jank2_x1, jank1_x2 + jank2_x2)

In [ ]:
a, b = symbols('a b')
leftover(a*jank1_x1 + b*jank2_x1,
         a*jank1_x2 + b*jank2_x2)

Not every pair of functions works: squaring a solution breaks it, because the equations are linear.

In [ ]:
leftover(jank1_x1**2, jank1_x2**2)

## 5. The special solutions: both carts at one frequency

Guess that both carts oscillate at the **same** frequency $\omega$, with amplitudes $C_1$ and $C_2$.
Plugging in turns each equation into (a number) $\times \cos(\omega t) = 0$, so the numbers must vanish.

In [ ]:
w = symbols('omega', positive=True)
C1, C2 = symbols('C_1 C_2')
eq1, eq2 = leftover(C1*cos(w*t), C2*cos(w*t))
eq1 = simplify(eq1 / cos(w*t))
eq2 = simplify(eq2 / cos(w*t))
eq1, eq2

$C_1 = C_2 = 0$ always works, and is boring: nothing moves. To get anything else, solve the first equation
for $C_2$, put it in the second, and ask which $\omega$ allow $C_1 \neq 0$.

In [ ]:
C2_from_eq1 = solve(eq1, C2)[0]
condition = factor(simplify(eq2.subs(C2, C2_from_eq1) / C1))
condition

In [ ]:
special_w = solve(condition, w)
[(ww, simplify(C2_from_eq1.subs(w, ww) / C1)) for ww in special_w]

Two frequencies work: $\omega_0$ with $C_2/C_1 = 1$ (the carts move together; the middle spring never
stretches) and $\sqrt{3}\,\omega_0$ with $C_2/C_1 = -1$ (mirror images). These are the **normal modes**.

## 6. From amounts of each mode to where to put the carts, and back

Every janky solution above is $A$ of mode 1 plus $B$ of mode 2, starting from rest:

$$x_1 = A\cos(\omega_0 t) + B\cos(\sqrt{3}\,\omega_0 t) \qquad x_2 = A\cos(\omega_0 t) - B\cos(\sqrt{3}\,\omega_0 t)$$

In [ ]:
A, B = symbols('A B')
x1_gen = A*cos(w0*t) + B*cos(sqrt(3)*w0*t)
x2_gen = A*cos(w0*t) - B*cos(sqrt(3)*w0*t)
leftover(x1_gen, x2_gen)

In [ ]:
start1, start2 = x1_gen.subs(t, 0), x2_gen.subs(t, 0)
start1, start2

In [ ]:
X1, X2 = symbols('X_1 X_2')
solve([Eq(start1, X1), Eq(start2, X2)], [A, B])

## 7. Watch it

Put the demo's numbers in the first two lines (these are placeholders until the carts are measured).
`A` and `B` are in centimetres; set one of them to 0 for a normal mode.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact

k_demo = 10.0   # N/m, placeholder
m_demo = 0.5    # kg, placeholder
w0_demo = np.sqrt(k_demo / m_demo)


def plot_carts(A=2.0, B=1.0):
    tt = np.linspace(0, 10, 1000)
    x1 = A*np.cos(w0_demo*tt) + B*np.cos(np.sqrt(3)*w0_demo*tt)
    x2 = A*np.cos(w0_demo*tt) - B*np.cos(np.sqrt(3)*w0_demo*tt)
    plt.figure(figsize=(9, 3.5))
    plt.plot(tt, x1, label='cart 1')
    plt.plot(tt, x2, label='cart 2')
    plt.xlabel('t (s)')
    plt.ylabel('x (cm)')
    plt.title(f'start: x1 = {A + B:.1f} cm, x2 = {A - B:.1f} cm')
    plt.legend()
    plt.show()


interact(plot_carts, A=(-3.0, 3.0, 0.5), B=(-3.0, 3.0, 0.5))

## 8. If one cart carries an extra weight

Put a weight on cart 2 so that $m_2 = 2 m_1$ (change the 2 below to the real ratio). The modes are no
longer "together" and "mirror image", but the recipe of section 5 still finds exactly two of them.

In [ ]:
ratio = 2
m1 = m
m2 = ratio*m
e1 = -m1*w**2*C1 - (-k*C1 + k*(C2 - C1))
e2 = -m2*w**2*C2 - (-k*C2 - k*(C2 - C1))
C2_from_e1 = solve(e1, C2)[0]
cond = factor(simplify(e2.subs(C2, C2_from_e1) / C1))
[(simplify(ww / w0), simplify(C2_from_e1.subs(w, ww) / C1))
 for ww in solve(cond, w)]

Each pair is (frequency in units of $\omega_0$, $C_2/C_1$). To start the system in one of these modes, pull
cart 1 out by any amount and cart 2 out by $C_2/C_1$ times that amount (a negative ratio means the other way).

## What we found

*(your notes here)*